In [ ]:
import requests
import pandas as pd
import time

# ① APIキーとURLの設定
API_KEY = "YOUR_API_KEY_HERE"
URL = "http://webservice.recruit.co.jp/hotpepper/gourmet/v1/"

# ② 比較したいターゲットブランド（10ブランド）
brands = [
    "スターバックス", "ドトール", "コメダ珈琲", "タリーズ", "星乃珈琲",
    "ルノアール", "上島珈琲店", "ベローチェ", "サンマルクカフェ", "プロント"
]

shop_data = []

# ③ 各ブランドの店舗データを取得（APIのループ処理）
for brand in brands:
    print(f"📡 {brand}のデータを収集・テキスト解析中...")
    params = {
        "key": API_KEY,
        "keyword": brand,
        "format": "json",
        "count": 50  # 各ブランド最大50店舗をサンプリング
    }

    response = requests.get(URL, params=params)
    if response.status_code == 200:
        shops = response.json()["results"]["shop"]

        for shop in shops:
            # 検索ノイズを省くため、店名にブランド名の先頭3文字が含まれるか確認
            if brand[:3] in shop["name"]:

                # お店のキャッチコピーとジャンルのキャッチコピーを結合
                catch_text = shop.get("catch", "") + shop.get("genre", {}).get("catch", "")

                # 辞書にデータを追加（特徴量エンジニアリング）
                shop_data.append({
                    "ブランド": brand,

                    # --- お客様目線のオリジナル変数（テキストマイニング） ---
                    "スイーツ充実": 1 if any(word in catch_text for word in ["スイーツ", "ケーキ", "パンケーキ", "デザート"]) else 0,
                    "本格コーヒー": 1 if any(word in catch_text for word in ["自家焙煎", "ドリップ", "本格", "抽出", "珈琲豆"]) else 0,
                    "食事・軽食": 1 if any(word in catch_text for word in ["サンドイッチ", "パスタ", "フード", "食事", "ランチ"]) else 0,
                    "モーニング": 1 if "モーニング" in catch_text else 0,
                    "ゆったり空間": 1 if any(word in catch_text for word in ["ゆったり", "ソファ", "くつろぎ", "落ち着いた", "リラックス"]) else 0,

                    # --- API標準の設備・サービス変数 ---
                    "深夜営業": 1 if '営業している' in shop.get("midnight", "") else 0,
                    "駐車場あり": 1 if 'あり' in shop.get("parking", "") else 0,
                    "個室あり": 1 if 'あり' in shop.get("private_room", "") else 0,
                })

    # サーバーに負荷をかけないよう1秒待機
    time.sleep(1)

# ④ 取得したデータをPandasのデータフレームに変換
df = pd.DataFrame(shop_data)

# ⑤ ブランドごとに各変数の「該当数」を合計してクロス集計表を作成
cross_table = df.groupby("ブランド").sum()

# ⑥ どのブランドも全く該当しなかった変数（合計0の列）を自動で削除
cross_table = cross_table.loc[:, (cross_table != 0).any(axis=0)]

print("\n🎉 データの前処理が完了しました！コレスポンデンス分析の入力表です：\n")
print(cross_table)

📡 スターバックスのデータを収集・テキスト解析中...
📡 ドトールのデータを収集・テキスト解析中...
📡 コメダ珈琲のデータを収集・テキスト解析中...
📡 タリーズのデータを収集・テキスト解析中...
📡 星乃珈琲のデータを収集・テキスト解析中...
📡 ルノアールのデータを収集・テキスト解析中...
📡 上島珈琲店のデータを収集・テキスト解析中...
📡 ベローチェのデータを収集・テキスト解析中...
📡 サンマルクカフェのデータを収集・テキスト解析中...
📡 プロントのデータを収集・テキスト解析中...

🎉 データの前処理が完了しました！コレスポンデンス分析の入力表です：

          スイーツ充実  本格コーヒー  食事・軽食  モーニング  ゆったり空間  深夜営業  駐車場あり  個室あり
ブランド                                                             
コメダ珈琲          0       0      0      0       2     1      4     0
サンマルクカフェ       0       1      2      0       0     0      2     0
スターバックス        0       0      0      0       0     0      9     0
タリーズ           0       1      0      1       1     0      4     0
ドトール           0       1      1      0       1     0      5     0
プロント           1       0      1      0       0     1      9     9
上島珈琲店          0       0      0      0       0     0      1     0
星乃珈琲           0      23      0      0       0     0      0     0


In [ ]:
!pip install prince

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.3/197.3 kB 5.2 MB/s eta 0:00:00


In [ ]:
import prince

print("📊 コレスポンデンス分析（次元削減）を実行します...")

# ① princeのコレスポンデンス分析（CA）モデルを準備
# n_components=2： 「2次元（X軸・Y軸）の影絵にしてね」という指示です
ca = prince.CA(
    n_components=2,
    n_iter=3,
    copy=True,
    check_input=True,
    engine='sklearn',
    random_state=42 # 毎回同じ計算結果（座標）になるように数値を固定
)

# ② クロス集計表をモデルに投入（ここで複雑な特異値分解が行われます！）
ca = ca.fit(cross_table)

# ③ ブランド（行）の2次元座標を抽出
brand_coords = ca.row_coordinates(cross_table)
brand_coords.columns = ['X', 'Y'] # わかりやすくX座標・Y座標と名前をつける

# ④ 特徴量（列）の2次元座標を抽出
feature_coords = ca.column_coordinates(cross_table)
feature_coords.columns = ['X', 'Y']

print("\n✅ 【ブランドの座標】")
print(brand_coords)

print("\n✅ 【特徴量（キーワード）の座標】")
print(feature_coords)

📊 コレスポンデンス分析（次元削減）を実行します...

✅ 【ブランドの座標】
                 X         Y
ブランド                        
コメダ珈琲    -0.665854 -0.815580
サンマルクカフェ -0.141566 -0.156183
スターバックス  -0.667527 -0.485323
タリーズ     -0.308768 -0.848698
ドトール     -0.360065 -0.525794
プロント     -0.770081  0.893377
上島珈琲店    -0.667527 -0.485323
星乃珈琲      1.445986  0.118676

✅ 【特徴量（キーワード）の座標】
               X         Y
スイーツ充実 -0.828926  1.433233
本格コーヒー  1.343335  0.073974
食事・軽食  -0.380318  0.022146
モーニング  -0.332362 -1.361555
ゆったり空間 -0.538353 -1.205483
深夜営業   -0.772831  0.062404
駐車場あり  -0.620140 -0.302516
個室あり   -0.828926  1.433233


In [ ]:
# 座標データをCSVファイルとして保存（ファイル名は何でもOKです）
brand_coords.to_csv("brand_coords.csv")
feature_coords.to_csv("feature_coords.csv")

In [ ]:
# 元データの確認

# ① APIキーとURLの設定
API_KEY = "YOUR_API_KEY_HERE"
URL = "http://webservice.recruit.co.jp/hotpepper/gourmet/v1/"

brands = [
    "スターバックス", "ドトール", "コメダ珈琲", "タリーズ", "星乃珈琲",
    "ルノアール", "上島珈琲店", "ベローチェ", "サンマルクカフェ", "プロント"
]

raw_text_data = []

for brand in brands:
    print(f"📡 {brand}の生テキストを取得中...")
    params = {
        "key": API_KEY,
        "keyword": brand,
        "format": "json",
        "count": 50
    }

    response = requests.get(URL, params=params)
    if response.status_code == 200:
        shops = response.json()["results"]["shop"]

        for shop in shops:
            if brand[:3] in shop["name"]:
                # APIから各種テキスト要素をそのまま引っ張ってくる
                catch = shop.get("catch", "")
                genre_catch = shop.get("genre", {}).get("catch", "")
                access = shop.get("access", "") # アクセス情報にも特徴が出ることがあります

                raw_text_data.append({
                    "ブランド": brand,
                    "店舗名": shop.get("name", ""),
                    "お店のキャッチコピー": catch,
                    "ジャンルのキャッチコピー": genre_catch,
                    # 分析用に結合したテキストも用意しておく
                    "結合テキスト": catch + genre_catch
                })

    time.sleep(1)

# データフレーム化してCSVに保存
df_raw = pd.DataFrame(raw_text_data)

# Excelで文字化けしないように utf-8-sig を指定して保存
df_raw.to_csv("cafe_raw_text.csv", index=False, encoding="utf-8-sig")

print("\n🎉 生データの抽出が完了しました！同じフォルダにある cafe_raw_text.csv を開いてみてください。")

📡 スターバックスの生テキストを取得中...
📡 ドトールの生テキストを取得中...
📡 コメダ珈琲の生テキストを取得中...
📡 タリーズの生テキストを取得中...
📡 星乃珈琲の生テキストを取得中...
📡 ルノアールの生テキストを取得中...
📡 上島珈琲店の生テキストを取得中...
📡 ベローチェの生テキストを取得中...
📡 サンマルクカフェの生テキストを取得中...
📡 プロントの生テキストを取得中...

🎉 生データの抽出が完了しました！同じフォルダにある cafe_raw_text.csv を開いてみてください。
